# VisionMirror CatVTON Google Colab GPU Server
This notebook installs CatVTON, starts a native FastAPI model server on GPU, tunnels it via Cloudflare Tunnel, and prints the public API URL.

In [ ]:
# 1. Verify GPU availability
import torch
assert torch.cuda.is_available(), "GPU is not enabled! Please go to Runtime -> Change runtime type and select GPU (T4/A100)."

In [ ]:
# 2. Clone repository & install dependencies
!git clone https://github.com/swarnaverma10/VisionMirror.git
%cd VisionMirror/catvton

# Install standard packages
!pip install -r requirements.txt
!pip install fastapi uvicorn python-multipart

In [ ]:
# 3. Run FastAPI and Cloudflare Tunnel
import subprocess
import time
import re
import sys

# Download Cloudflare Tunnel client
print("Downloading cloudflared...")
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

# Start FastAPI server in the background
print("Starting FastAPI server...")
server_proc = subprocess.Popen(["uvicorn", "colab_server:app", "--host", "127.0.0.1", "--port", "8000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

# Start Cloudflare Tunnel in the background
print("Starting Cloudflare Tunnel...")
tunnel_proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

time.sleep(5)  # Wait for startup

# Listen to tunnel output and extract the public URL
public_url = None
try:
    for line in iter(tunnel_proc.stdout.readline, ""):
        sys.stdout.write(line)
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            public_url = match.group(0)
            print("\n" + "="*50)
            print(f"PUBLIC API URL: {public_url}")
            print("="*50 + "\n")
            break
except KeyboardInterrupt:
    print("Stopping...")
    server_proc.terminate()
    tunnel_proc.terminate()

# Keep running to stream logs
try:
    while True:
        line = server_proc.stdout.readline()
        if line:
            sys.stdout.write(line)
        time.sleep(0.1)
except KeyboardInterrupt:
    print("Stopping server...")
    server_proc.terminate()
    tunnel_proc.terminate()